In [1]:

import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


data1 = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_hsv_features.npz")
X1_train = data1["X_train"]
y1_train = data1["y_train"]
X1_test = data1["X_test"]
y1_test = data1["y_test"]

data2 = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_ycrcb_features.npz")
X2_train = data2["X_train"]
y2_train = data2["y_train"]
X2_test = data2["X_test"]
y2_test = data2["y_test"]

data = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_hsv_features.npz")

print(X1_train.shape, X2_train.shape)

X_train = np.concatenate((X1_train, X2_train), axis=1)
y_train = y1_train
X_test = np.concatenate((X1_test, X2_test), axis=1)
y_test = y1_test

# X_train = data["X_train"]
# y_train = data["y_train"]
# X_test = data["X_test"]
# y_test = data["y_test"]

print(X_train.shape, X_test.shape)


# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training DNN...")

model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")  # binary classification
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1
)

print("Evaluating...")

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fnr = 1 - tpr

idx = np.nanargmin(np.abs(fnr - fpr))
eer = (fpr[idx] + fnr[idx]) / 2

print("EER:", eer)

import matplotlib.pyplot as plt

plt.figure()

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train Loss", "Validation Loss"])

plt.show()

ModuleNotFoundError: No module named 'tensorflow'